# 12 — OE3: padrões sazonais e calendário comercial em 2017

**Objetivo (OE3, Parte 1):** analisar de forma exploratória os padrões sazonais da receita, com ênfase na Black Friday de 2017 e nos feriados nacionais, e identificar os eventos do calendário com maior associação com a receita.  
**Lê:** histórico do treino (`panel_base` até `TRAIN_TARGET_END`), `calendario_eventos`, `perfis_treino`, treino e `selecao_modelo.json`. **Não lê o teste.**  
**Grava:** `oe3_*.csv` e `oe3_receita_semanal_2017.png`.

**Lift** = receita da semana ÷ média das 4 semanas anteriores à janela do evento, pulando semanas de outros eventos comerciais e a semana seguinte à Black Friday. Como a coorte cresce em 2017, cada lift também é comparado com a mediana dos lifts das semanas comuns. Cada evento ocorre uma vez: os resultados são **associações observadas**, não efeitos causais.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "12_oe3_sazonalidade"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import sazonalidade as SZ, graficos

ANO = 2017
# Janelas dos eventos: (rótulo, chave no calendário, semanas ANTES do evento na janela). Ajuste à Tabela 3 da Parte 1.
EVENTOS = SZ.EVENTOS_PADRAO
hist = A.carregar_historico_treino()
cal = A.carregar_df("calendario_eventos")
perfis = A.carregar_df("perfis_treino")
tot, ativos = SZ.receita_semanal(hist, C.PANEL_START, C.TRAIN_TARGET_END)
print(f"{len(tot)} semanas: {tot.index.min().date()} a {(tot.index.max() + pd.Timedelta(days=6)).date()}")

## 1. Padrão sazonal geral

In [ ]:
jan = SZ.janelas_eventos(cal, ANO, EVENTOS)
fer = SZ.semanas_feriado(cal, ANO)
fig, ax = plt.subplots(figsize=(12, 4.6))
for _, r in jan.drop_duplicates("evento").iterrows():
    w0 = r["primeira_semana_janela"]; w1 = SZ.segunda(pd.Timestamp(r["data"]))
    ax.axvspan(w0 - pd.Timedelta(days=3), w1 + pd.Timedelta(days=4), color="#eb6834", alpha=0.13, lw=0)
    ax.text(w1, tot.max() * 1.55, r["evento"], rotation=90, va="top", ha="center", fontsize=8, color=graficos.INK2)
ax.plot(tot.index, tot.values, color=graficos.C_CHOSEN, lw=2, marker="o", ms=3, label="Receita semanal da coorte")
yf = tot.reindex(sorted(fer["semana"].unique())).dropna()
ax.scatter(yf.index, [tot.max() * 0.03] * len(yf), marker="|", s=120, color=graficos.INK, label="Semana com feriado nacional")
ax.set_ylim(0, tot.max() * 1.6)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:,.0f} mil"))
ax.set_ylabel("R$ por semana"); graficos.clean(ax)
ax.legend(frameon=False, loc="upper left", fontsize=8)
ax.set_title("Receita semanal da coorte em 2017 (treino), com eventos comerciais (faixas) e feriados (marcas)", loc="left", fontsize=10)
fig.tight_layout(); A.figura(fig, "oe3_receita_semanal_2017.png"); plt.show()

mensal = pd.DataFrame({"receita_R$": tot.groupby(tot.index.to_period("M")).sum(),
                       "vendedores_no_painel": ativos.groupby(ativos.index.to_period("M")).max()})
mensal["receita_por_vendedor_R$"] = mensal["receita_R$"] / mensal["vendedores_no_painel"]
mensal.index = mensal.index.astype(str)
A.tabela(mensal.reset_index(names="mes"), "oe3_receita_mensal_2017.csv"); display(mensal)
print("Meses mais fortes (receita por vendedor):", list(mensal["receita_por_vendedor_R$"].nlargest(3).index))
print("Meses mais fracos (receita por vendedor):", list(mensal["receita_por_vendedor_R$"].nsmallest(3).index))
print("A receita total cresce também porque vendedores entram na coorte ao longo do ano: veja vendedores_no_painel.")

## 2. Lift por evento comercial (resposta do OE3: ordem de associação)

In [ ]:
lift, ref = SZ.tabela_lift(tot, cal, ANO, EVENTOS)
A.tabela(lift, "oe3_lift_eventos.csv"); display(lift)
print(f"Referência: lift mediano das semanas comuns = {ref['lift_mediano_semanas_comuns']:.3f} ({ref['n_semanas_comuns']} semanas). "
      "lift_vs_semana_comum > 1 indica receita acima do padrão de uma semana sem evento.")

## 3. Black Friday 2017 (24/11; semana de 20 a 26/11)

In [ ]:
data_bf = cal.loc[(cal["ano"] == ANO) & (cal["evento"] == "commercial_black_friday"), "date"].min()
bf, base_bf = SZ.janela_black_friday(tot, data_bf, antes=2, depois=1, excluir=SZ.semanas_excluidas(cal, ANO, EVENTOS))
A.tabela(bf, "oe3_black_friday_janela.csv"); display(bf)
print(f"Base: média das 4 semanas antes da janela = R$ {base_bf:,.2f}")
bfv = SZ.black_friday_vendedores(hist, perfis, data_bf)
A.tabela(bfv, "oe3_black_friday_vendedores.csv"); display(bfv)

## 4. Feriados nacionais (agrupados)

In [ ]:
tab_fer, res_fer = SZ.tabela_feriados(tot, cal, ANO, EVENTOS)
A.tabela(tab_fer, "oe3_feriados.csv"); display(tab_fer)
print(pd.Series(res_fer).round(3))
_ = A.salvar_json({"lift_referencia": ref, "feriados": res_fer, "base_black_friday": base_bf}, "oe3_resumo")

## 5. Ligação com a modelagem: peso do calendário no modelo escolhido

In [ ]:
from tcc_fluxo import Especificacao
from tcc_fluxo.modelos import ajustar
spec_json = A.carregar_json("painel_spec"); spec = Especificacao.de_json(spec_json, C)
MODELO = A.carregar_json("selecao_modelo")["modelo_escolhido"]
prever = ajustar(MODELO, A.carregar_treino(), spec)          # mesmo ajuste do notebook 05 (só treino)
grupos, detalhe = SZ.importancia_grupos(prever, spec_json)
if grupos is None:
    print(f"Importância não disponível para {MODELO}.")
else:
    A.tabela(grupos, "oe3_importancia_grupos.csv"); A.tabela(detalhe, "oe3_importancia_atributos.csv")
    print(f"Importância por grupo — {MODELO}:"); display(grupos)
    print("Atributos de calendário mais usados:"); display(detalhe[detalhe["grupo"].str.startswith("calend")].head(8))

### Para o texto (seção 5.6)
- **Figura:** `oe3_receita_semanal_2017.png`. **Tabela:** `oe3_lift_eventos.csv` (já ordenada pelo lift).
- Escrever "associação observada", não "efeito": cada evento ocorre uma única vez em 2017.
- Em 2017, Natal (25/12, segunda) e Réveillon (31/12, domingo) caem na **mesma** semana; a semana anterior é a de compras de Natal.
- Dia dos Namorados (12/06) e Corpus Christi (15/06) também dividem a semana: a coluna `feriados_na_mesma_semana` mostra essas sobreposições.

In [ ]:
A.finalizar()